### 📂 Dataset Setup
This project utilizes the **PlantVillage Dataset** for training and evaluation. Since the dataset size is around 2 GB, it is not included in this GitHub repository.

To run this notebook successfully:
1. Download the dataset from Kaggle: [PlantVillage Dataset on Kaggle]
2. Extract the downloaded zip file on your local machine.
3. Locate the `color` folder inside the dataset.
4. Update the `data_dir` path in the code cell below to point to your local `color` folder directory.

In [ ]:
import tensorflow as tf

data_dir = r"YOUR_DATASET_PATH_HERE/plantvillage dataset/color"


train_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(256, 256),
    batch_size=16
)


exam_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(256, 256),
    batch_size=16
)


total_batches = tf.data.experimental.cardinality(exam_data)



autotune = tf.data.AUTOTUNE


train_data = train_data.cache().shuffle(1000).prefetch(buffer_size=autotune)


test_data = exam_data.take(total_batches // 2).cache().prefetch(buffer_size=autotune)


validation_data = exam_data.skip(total_batches // 2).cache().prefetch(buffer_size=autotune)



Found 54305 files belonging to 38 classes.
Using 43444 files for training.
Found 54305 files belonging to 38 classes.
Using 10861 files for validation.


In [2]:
from tensorflow.keras.layers import (
Dense,
Conv2D,
GlobalAveragePooling2D,
MaxPooling2D,
Dropout,
Rescaling,
Flatten,
BatchNormalization,
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import  Sequential
from tensorflow.keras.callbacks import EarlyStopping




model = Sequential([

    Rescaling(1./255, input_shape=(256, 256, 3)),



    Conv2D(32, (3, 3), activation='swish', kernel_regularizer=l2(0.0001)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Dropout(0.2),


    Conv2D(64, (3, 3), activation='swish',kernel_regularizer=l2(0.0001)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),



    Conv2D(64, (3, 3), activation='swish',kernel_regularizer=l2(0.0001)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),



    GlobalAveragePooling2D(),
    




    Dense(38, activation='softmax')
])

model.summary()




Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 256, 256, 3)       0         
                                                                 
 conv2d (Conv2D)             (None, 254, 254, 32)      896       
                                                                 
 batch_normalization (BatchN  (None, 254, 254, 32)     128       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 127, 127, 32)     0         
 )                                                               
                                                                 
 dropout (Dropout)           (None, 127, 127, 32)      0         
                                                                 
 conv2d_1 (Conv2D)           (None, 125, 125, 64)      1

In [3]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])





early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)


model.fit(
    train_data,
    validation_data=validation_data,
    epochs=100,
    callbacks=[early_stop]
)

Epoch 1/100
2716/2716 [==============================] - 582s 207ms/step - loss: 1.2427 - accuracy: 0.6664 - val_loss: 0.7938 - val_accuracy: 0.7565
Epoch 2/100
2716/2716 [==============================] - 619s 228ms/step - loss: 0.5946 - accuracy: 0.8341 - val_loss: 0.7693 - val_accuracy: 0.7668
Epoch 3/100
2716/2716 [==============================] - 623s 229ms/step - loss: 0.4232 - accuracy: 0.8846 - val_loss: 0.7471 - val_accuracy: 0.7846
Epoch 4/100
2716/2716 [==============================] - 623s 230ms/step - loss: 0.3490 - accuracy: 0.9070 - val_loss: 1.1778 - val_accuracy: 0.6989
Epoch 5/100
2716/2716 [==============================] - 624s 230ms/step - loss: 0.2996 - accuracy: 0.9235 - val_loss: 0.4701 - val_accuracy: 0.8611
Epoch 6/100
2716/2716 [==============================] - 625s 230ms/step - loss: 0.2701 - accuracy: 0.9334 - val_loss: 0.4455 - val_accuracy: 0.8698
Epoch 7/100
2716/2716 [==============================] - 627s 231ms/step - loss: 0.2455 - accuracy: 0.9399

In [4]:
print("evaluation model")

test_loss, test_acc = model.evaluate(test_data, verbose=2)

print(f'test accuracy : {test_acc*100:.2f}%')
print(f'test lose : {test_loss:.4f}')

evaluation model
339/339 - 14s - loss: 0.2190 - accuracy: 0.9554 - 14s/epoch - 42ms/step
test accuracy : 95.54%
test lose : 0.2190


In [5]:
model.save("model.keras")

print("model saved")

model saved
